In [ ]:
import numpy as np 
import scipy
import h5py
import matplotlib.pyplot as plt
import pandas as pd

from netneurotools.networks import struct_consensus, binarize_network

In [ ]:
matlab_conns_path = "/Users/adrian/Documents/01_projects/14_4D_lab/connectome_distances/data/raw/lexis_data/connectomes.mat"
age_path = "/Users/adrian/Documents/01_projects/14_4D_lab/connectome_distances/data/raw/lexis_data/age.mat"
ids_path = "/Users/adrian/Documents/01_projects/14_4D_lab/connectome_distances/data/raw/lexis_data/ids.csv"

In [ ]:
conns_mat = h5py.File(matlab_conns_path)
age_mat = h5py.File(age_path)
ids_df = pd.read_csv(ids_path)

In [ ]:
ids_df

In [ ]:
# Get all row indices of the rows that do not start with "HC"
non_hc_indices = ids_df[~ids_df['HCD0001305'].str.startswith('HC')].index
non_hc_indices.tolist()

In [ ]:
conns_array = np.array(conns_mat['connectomes']).T
conns_array_young_adults = conns_array[non_hc_indices, :, :]
conns_array.shape, conns_array_young_adults.shape

In [ ]:
ages = np.array(age_mat["age"])[0] # .keys()
plt.scatter(np.arange(len(ages)), ages, label='All Participants')
plt.scatter(non_hc_indices, ages[non_hc_indices], color='orange', s=5, label='Young Adults (Non-HC)')

In [ ]:
min(ages[non_hc_indices]), max(ages[non_hc_indices])

In [ ]:
plt.imshow(conns_array[0], cmap='hot', interpolation='nearest')

In [ ]:
distance_path = "/Users/adrian/Documents/01_projects/14_4D_lab/connectome_distances/data/preprocessed/lexis_data/02_distance_matrices/distance_matrix_100.npy"
d = np.load(distance_path)
d.shape

In [ ]:
hemiid = [0]*50 + [1]*50  # assuming 100 regions, first 50 left, next 50 right
hemiid = np.array(hemiid).reshape(-1,1)

# Only use young adults

In [ ]:
conns_array_young_adults.shape

In [ ]:
consensus_non_thres = struct_consensus(conns_array_young_adults.T, d, hemiid=hemiid, weighted=True) 
consensus = binarize_network(consensus_non_thres, retain=10).reshape(-1,100,100)
plt.imshow(consensus[0]) # , cmap='gray', interpolation='nearest')

In [ ]:
conns_young_adults_binarized = [] 

for A in conns_array_young_adults: 
    conns_young_adults_binarized.append(binarize_network(A, retain=10)) # .reshape(-1,100,100))
    
conns_young_adults_binarized = np.array(conns_young_adults_binarized)
conns_young_adults_binarized.shape

In [ ]:
plt.imshow(conns_young_adults_binarized[0])

In [ ]:
base_path = "/Users/adrian/Documents/01_projects/14_4D_lab/connectome_distances/data/preprocessed/lexis_data/01_connectomes/"
save_path = base_path + "00_connectomes_density10.npy" # i.e. consensus
np.save(save_path, consensus)
save_path = base_path + "00_individual_connectomes_bin.npy"
np.save(save_path, conns_young_adults_binarized)